In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/Base.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df["fraud_bool"].value_counts()

In [ ]:
df.describe().T

In [ ]:
df["fraud_bool"].value_counts(normalize=True) * 100

In [ ]:
df["fraud_bool"].value_counts(normalize=True)

In [ ]:
sentinel_columns = [
    "prev_address_months_count",
    "current_address_months_count",
    "bank_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w",
]

for column in sentinel_columns:
    print(column)
    print((df[column] == -1).sum())
    print((df[column] == -1).mean() * 100)
    print()

In [ ]:
df.groupby("fraud_bool")["prev_address_months_count"].apply(
    lambda s: (s == -1).mean() * 100
)

In [ ]:
df.groupby("fraud_bool")["bank_months_count"].apply(
    lambda s: (s == -1).mean() * 100
)

In [ ]:
df.assign(
    prev_address_missing=df["prev_address_months_count"] == -1
).groupby("prev_address_missing")["fraud_bool"].mean() * 100

In [ ]:
df.assign(
    bank_months_missing=df["bank_months_count"] == -1
).groupby("bank_months_missing")["fraud_bool"].mean() * 100

In [ ]:
df.groupby("month")["fraud_bool"].agg(
    count="count",
    fraud_count="sum",
    fraud_rate="mean"
)

In [ ]:
monthly_fraud = df.groupby("month")["fraud_bool"].mean() * 100
monthly_fraud

In [ ]:
df.groupby("month")[
    ["income", "credit_risk_score", "velocity_6h"]
].mean()

In [ ]:
df.groupby("month")[
    ["income", "credit_risk_score", "velocity_6h"]
].median()

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns

month_0 = df[df["month"] == 0][numeric_columns].mean()
month_7 = df[df["month"] == 7][numeric_columns].mean()

In [ ]:
comparison = pd.DataFrame({
    "month_0_mean": month_0,
    "month_7_mean": month_7
})

In [ ]:
comparison["absolute_change"] = (
    comparison["month_7_mean"] - comparison["month_0_mean"]
)

In [ ]:
comparison.head()

In [ ]:
comparison["relative_change_pct"] = (
    (
        comparison["month_7_mean"]
        - comparison["month_0_mean"]
    )
    / comparison["month_0_mean"].abs()
) * 100

In [ ]:
comparison.head()

In [ ]:
feature_comparison = comparison.drop(
    index=["fraud_bool", "month"],
    errors="ignore"
)

feature_comparison["abs_relative_change_pct"] = (
    feature_comparison["relative_change_pct"].abs()
)

feature_comparison.sort_values(
    "abs_relative_change_pct",
    ascending=False
).head(10)

In [ ]:
features_to_check = [
    "zip_count_4w",
    "velocity_6h"
]

df[df["month"].isin([0, 7])].groupby("month")[features_to_check].quantile(
    [0.25, 0.5, 0.75]
)

In [ ]:
import matplotlib.pyplot as plt

month_0_velocity = df[df["month"] == 0]["velocity_6h"]
month_7_velocity = df[df["month"] == 7]["velocity_6h"]

plt.figure(figsize=(10, 6))

plt.hist(
    month_0_velocity,
    bins=50,
    alpha=0.5,
    label="Month 0"
)

plt.hist(
    month_7_velocity,
    bins=50,
    alpha=0.5,
    label="Month 7"
)

plt.xlabel("velocity_6h")
plt.ylabel("Frequency")
plt.title("Distribution of velocity_6h: Month 0 vs Month 7")
plt.legend()

plt.show()

In [ ]:
expected = df[df["month"] == 0]["velocity_6h"]
actual = df[df["month"] == 7]["velocity_6h"]

In [ ]:
import pandas as pd

_, bins = pd.qcut(
    expected,
    q=10,
    retbins=True,
    duplicates="drop"
)

In [ ]:
expected_bins = pd.cut(
    expected,
    bins=bins,
    include_lowest=True
)

actual_bins = pd.cut(
    actual,
    bins=bins,
    include_lowest=True
)

In [ ]:
expected_pct = expected_bins.value_counts(
    normalize=True,
    sort=False
)

actual_pct = actual_bins.value_counts(
    normalize=True,
    sort=False
)

In [ ]:
psi_table = pd.DataFrame({
    "expected_pct": expected_pct,
    "actual_pct": actual_pct
})

psi_table

In [ ]:
import numpy as np

psi_table["psi_contribution"] = (
    (psi_table["actual_pct"] - psi_table["expected_pct"])
    * np.log(
        psi_table["actual_pct"] / psi_table["expected_pct"]
    )
)

In [ ]:
psi = psi_table["psi_contribution"].sum()

psi

In [ ]:
from scipy.stats import ks_2samp

ks_statistic, p_value = ks_2samp(expected, actual)

print("KS statistic:", ks_statistic)
print("p-value:", p_value)

In [ ]:
import numpy as np
import pandas as pd

def calculate_psi(expected, actual, bins=10):
    _, bin_edges = pd.qcut(
        expected,
        q=bins,
        retbins=True,
        duplicates="drop"
    )

    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    expected_bins = pd.cut(
        expected,
        bins=bin_edges,
        include_lowest=True
    )

    actual_bins = pd.cut(
        actual,
        bins=bin_edges,
        include_lowest=True
    )

    expected_pct = expected_bins.value_counts(
        normalize=True,
        sort=False
    )

    actual_pct = actual_bins.value_counts(
        normalize=True,
        sort=False
    )

    epsilon = 1e-6

    expected_pct = expected_pct.clip(lower=epsilon)
    actual_pct = actual_pct.clip(lower=epsilon)

    psi = (
        (actual_pct - expected_pct)
        * np.log(actual_pct / expected_pct)
    ).sum()

    return psi

In [ ]:
from scipy.stats import ks_2samp

def calculate_ks(expected, actual):
    ks_statistic, p_value = ks_2samp(expected, actual)

    return ks_statistic, p_value

In [ ]:
excluded_columns = ["fraud_bool", "month"]

In [ ]:
numeric_features = [
    column
    for column in df.select_dtypes(include="number").columns
    if column not in excluded_columns
]

In [ ]:
drift_results = []

for feature in numeric_features:
    expected = df[df["month"] == 0][feature]
    actual = df[df["month"] == 7][feature]

    psi = calculate_psi(expected, actual)
    ks_statistic, p_value = calculate_ks(expected, actual)

    drift_results.append({
        "feature": feature,
        "psi": psi,
        "ks_statistic": ks_statistic,
        "p_value": p_value
    })

In [ ]:
drift_df = pd.DataFrame(drift_results)

In [ ]:
drift_df.sort_values(
    "psi",
    ascending=False
).head(10)

In [ ]:
categorical_features = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

categorical_features

In [ ]:
feature = "payment_type"

In [ ]:
expected_pct = (
    df[df["month"] == 0][feature]
    .value_counts(normalize=True)
)

In [ ]:
actual_pct = (
    df[df["month"] == 7][feature]
    .value_counts(normalize=True)
)

In [ ]:
category_comparison = pd.DataFrame({
    "expected_pct": expected_pct,
    "actual_pct": actual_pct
}).fillna(0)

category_comparison

In [ ]:
epsilon = 1e-6

category_comparison["expected_pct"] = (
    category_comparison["expected_pct"].clip(lower=epsilon)
)

category_comparison["actual_pct"] = (
    category_comparison["actual_pct"].clip(lower=epsilon)
)

category_comparison["psi_contribution"] = (
    (category_comparison["actual_pct"] - category_comparison["expected_pct"])
    * np.log(
        category_comparison["actual_pct"]
        / category_comparison["expected_pct"]
    )
)

category_psi = category_comparison["psi_contribution"].sum()

category_psi

In [ ]:
category_comparison

In [ ]:
def calculate_categorical_psi(expected, actual):
    expected_pct = expected.value_counts(normalize=True)
    actual_pct = actual.value_counts(normalize=True)

    comparison = pd.DataFrame({
        "expected_pct": expected_pct,
        "actual_pct": actual_pct
    }).fillna(0)

    epsilon = 1e-6

    comparison["expected_pct"] = comparison["expected_pct"].clip(lower=epsilon)
    comparison["actual_pct"] = comparison["actual_pct"].clip(lower=epsilon)

    comparison["psi_contribution"] = (
        (comparison["actual_pct"] - comparison["expected_pct"])
        * np.log(
            comparison["actual_pct"] / comparison["expected_pct"]
        )
    )

    return comparison["psi_contribution"].sum()

In [ ]:
categorical_results = []

for feature in categorical_features:
    expected = df[df["month"] == 0][feature]
    actual = df[df["month"] == 7][feature]

    psi = calculate_categorical_psi(expected, actual)

    categorical_results.append({
        "feature": feature,
        "psi": psi
    })

categorical_drift_df = pd.DataFrame(categorical_results)

categorical_drift_df.sort_values(
    "psi",
    ascending=False
)

In [ ]:
train_df = df[df["month"].between(0, 5)].copy()
val_df = df[df["month"] == 6].copy()
test_df = df[df["month"] == 7].copy()

In [ ]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("Train fraud rate:", train_df["fraud_bool"].mean())
print("Validation fraud rate:", val_df["fraud_bool"].mean())
print("Test fraud rate:", test_df["fraud_bool"].mean())

In [ ]:
feature_columns_to_drop = ["fraud_bool", "month"]

X_train = train_df.drop(columns=feature_columns_to_drop)
y_train = train_df["fraud_bool"]

X_val = val_df.drop(columns=feature_columns_to_drop)
y_val = val_df["fraud_bool"]

X_test = test_df.drop(columns=feature_columns_to_drop)
y_test = test_df["fraud_bool"]

In [ ]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    exclude=["object", "string"]
).columns.tolist()

print("Categorical:", categorical_features)
print("Numeric count:", len(numeric_features))
print("Categorical count:", len(categorical_features))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
baseline_model.fit(X_train, y_train)

In [ ]:
y_val_pred = baseline_model.predict(X_val)

In [ ]:
y_val_proba = baseline_model.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

print("Confusion matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("Precision:", precision_score(y_val, y_val_pred))
print("Recall:", recall_score(y_val, y_val_pred))
print("F1:", f1_score(y_val, y_val_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_val_proba))
print("PR-AUC:", average_precision_score(y_val, y_val_proba))

In [ ]:
thresholds = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]

results = []

for threshold in thresholds:
    y_pred_threshold = (y_val_proba >= threshold).astype(int)

    precision = precision_score(y_val, y_pred_threshold)
    recall = recall_score(y_val, y_pred_threshold)
    f1 = f1_score(y_val, y_pred_threshold)

    alerts = y_pred_threshold.sum()

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "alerts": alerts
    })

threshold_results = pd.DataFrame(results)

threshold_results

In [ ]:
import numpy as np

thresholds = np.arange(0.50, 1.00, 0.01)

results = []

for threshold in thresholds:
    y_pred_threshold = (y_val_proba >= threshold).astype(int)

    precision = precision_score(y_val, y_pred_threshold)
    recall = recall_score(y_val, y_pred_threshold)
    f1 = f1_score(y_val, y_pred_threshold)

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "alerts": y_pred_threshold.sum()
    })

threshold_results_fine = pd.DataFrame(results)

In [ ]:
threshold_results_fine.loc[
    threshold_results_fine["f1"].idxmax()
]

In [ ]:
business_candidates = threshold_results_fine[
    threshold_results_fine["precision"] >= 0.20
]

best_business_threshold = business_candidates.loc[
    business_candidates["recall"].idxmax()
]

best_business_threshold

In [ ]:
y_test_proba = baseline_model.predict_proba(X_test)[:, 1]

In [ ]:
chosen_threshold = 0.91

y_test_pred = (y_test_proba >= chosen_threshold).astype(int)

In [ ]:
print("Confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("F1:", f1_score(y_test, y_test_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print("PR-AUC:", average_precision_score(y_test, y_test_proba))
print("Alerts:", y_test_pred.sum())

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative:", negative_count)
print("Positive:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("classifier", xgb_model)
    ]
)

In [ ]:
xgb_pipeline.fit(X_train, y_train)

In [ ]:
y_val_xgb_proba = xgb_pipeline.predict_proba(X_val)[:, 1]
y_val_xgb_pred = (y_val_xgb_proba >= 0.5).astype(int)

In [ ]:
print("Confusion matrix:")
print(confusion_matrix(y_val, y_val_xgb_pred))

print("Precision:", precision_score(y_val, y_val_xgb_pred))
print("Recall:", recall_score(y_val, y_val_xgb_pred))
print("F1:", f1_score(y_val, y_val_xgb_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_val_xgb_proba))
print("PR-AUC:", average_precision_score(y_val, y_val_xgb_proba))

In [ ]:
thresholds = np.arange(0.50, 1.00, 0.01)

xgb_threshold_results = []

for threshold in thresholds:
    y_pred = (y_val_xgb_proba >= threshold).astype(int)

    xgb_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, y_pred),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "alerts": y_pred.sum()
    })

xgb_threshold_df = pd.DataFrame(xgb_threshold_results)

In [ ]:
xgb_threshold_df.loc[
    xgb_threshold_df["f1"].idxmax()
]

In [ ]:
xgb_business_candidates = xgb_threshold_df[
    xgb_threshold_df["precision"] >= 0.20
]

xgb_best_business_threshold = xgb_business_candidates.loc[
    xgb_business_candidates["recall"].idxmax()
]

xgb_best_business_threshold

In [ ]:
chosen_threshold_xgb = 0.90

In [ ]:
y_test_xgb_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

chosen_threshold_xgb = 0.90

y_test_xgb_pred = (
    y_test_xgb_proba >= chosen_threshold_xgb
).astype(int)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_test_xgb_pred))

print("Precision:", precision_score(y_test, y_test_xgb_pred))
print("Recall:", recall_score(y_test, y_test_xgb_pred))
print("F1:", f1_score(y_test, y_test_xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_test_xgb_proba))
print("PR-AUC:", average_precision_score(y_test, y_test_xgb_proba))
print("Alerts:", y_test_xgb_pred.sum())

In [ ]:
import torch
import torch.nn as nn

class FraudMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print(X_train_processed.shape)
print(X_val_processed.shape)
print(X_test_processed.shape)

In [ ]:
input_dim = X_train_processed.shape[1]

model = FraudMLP(input_dim)

model

In [ ]:
import torch

In [ ]:
X_train_tensor = torch.from_numpy(
    X_train_processed.astype("float32", copy=False)
)

y_train_tensor = torch.from_numpy(
    y_train.to_numpy(dtype="float32")
).unsqueeze(1)

X_val_tensor = torch.from_numpy(
    X_val_processed.astype("float32", copy=False)
)

y_val_tensor = torch.from_numpy(
    y_val.to_numpy(dtype="float32")
).unsqueeze(1)

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
df = pd.read_csv("../data/raw/Base.csv")

In [ ]:
train_df = df[df["month"].between(0, 5)].copy()
val_df = df[df["month"] == 6].copy()

X_train = train_df.drop(columns=["fraud_bool", "month"])
y_train = train_df["fraud_bool"]

X_val = val_df.drop(columns=["fraud_bool", "month"])
y_val = val_df["fraud_bool"]

In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    exclude=["object", "string"]
).columns.tolist()

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [ ]:
preprocessor.fit(X_train)

In [ ]:
X_train_processed = preprocessor.transform(X_train).astype(
    np.float32,
    copy=False
)

print(X_train_processed.shape)
print(X_train_processed.dtype)

In [ ]:
X_train_tensor = torch.from_numpy(X_train_processed)

y_train_tensor = torch.from_numpy(
    y_train.to_numpy(dtype=np.float32)
).unsqueeze(1)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True
)

In [ ]:
X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

In [ ]:
X_val_processed = preprocessor.transform(X_val).astype(
    np.float32,
    copy=False
)

X_val_tensor = torch.from_numpy(X_val_processed)

y_val_tensor = torch.from_numpy(
    y_val.to_numpy(dtype=np.float32)
).unsqueeze(1)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

val_loader = DataLoader(
    val_dataset,
    batch_size=512,
    shuffle=False
)

In [ ]:
X_batch, y_batch = next(iter(val_loader))

print(X_batch.shape)
print(y_batch.shape)

In [ ]:
import torch.nn as nn

class FraudMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
model = FraudMLP(input_dim=51)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    # TRAIN
    model.train()
    total_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    # VALIDATION
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_val_loss += loss.item()

    average_val_loss = total_val_loss / len(val_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"- Train Loss: {average_train_loss:.4f} "
        f"- Val Loss: {average_val_loss:.4f}"
    )

In [ ]:
positive_count = (y_train == 1).sum()
negative_count = (y_train == 0).sum()

pos_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32
)

print(pos_weight)

In [ ]:
model = FraudMLP(input_dim=51)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [ ]:
import copy

In [ ]:
num_epochs = 30
patience = 3

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

for epoch in range(num_epochs):
    # TRAIN
    model.train()
    total_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    # VALIDATION
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_val_loss += loss.item()

    average_val_loss = total_val_loss / len(val_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"- Train Loss: {average_train_loss:.4f} "
        f"- Val Loss: {average_val_loss:.4f}"
    )

    # EARLY STOPPING
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

model.load_state_dict(best_model_state)

In [ ]:
model.eval()

val_probs = []
val_targets = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        logits = model(X_batch)

        probs = torch.sigmoid(logits)

        val_probs.append(probs.cpu())
        val_targets.append(y_batch.cpu())

val_probs = torch.cat(val_probs).numpy().ravel()
val_targets = torch.cat(val_targets).numpy().ravel()

In [ ]:
val_preds = (val_probs >= 0.5).astype(int)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

In [ ]:
print("Confusion matrix:")
print(confusion_matrix(val_targets, val_preds))

print("Precision:", precision_score(val_targets, val_preds))
print("Recall:", recall_score(val_targets, val_preds))
print("F1:", f1_score(val_targets, val_preds))
print("ROC-AUC:", roc_auc_score(val_targets, val_probs))
print("PR-AUC:", average_precision_score(val_targets, val_probs))

In [ ]:
thresholds = np.arange(0.50, 1.00, 0.01)

pytorch_threshold_results = []

for threshold in thresholds:
    preds = (val_probs >= threshold).astype(int)

    pytorch_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(val_targets, preds),
        "recall": recall_score(val_targets, preds),
        "f1": f1_score(val_targets, preds),
        "alerts": preds.sum()
    })

pytorch_threshold_df = pd.DataFrame(pytorch_threshold_results)

In [ ]:
pytorch_threshold_df.loc[
    pytorch_threshold_df["f1"].idxmax()
]

In [ ]:
pytorch_business_candidates = pytorch_threshold_df[
    pytorch_threshold_df["precision"] >= 0.20
]

pytorch_best_business_threshold = pytorch_business_candidates.loc[
    pytorch_business_candidates["recall"].idxmax()
]

pytorch_best_business_threshold

In [ ]:
chosen_threshold_pytorch = 0.93

In [ ]:
test_df = df[df["month"] == 7].copy()

X_test = test_df.drop(columns=["fraud_bool", "month"])
y_test = test_df["fraud_bool"]

In [ ]:
X_test_processed = preprocessor.transform(X_test).astype(
    np.float32,
    copy=False
)

X_test_tensor = torch.from_numpy(X_test_processed)

y_test_tensor = torch.from_numpy(
    y_test.to_numpy(dtype=np.float32)
).unsqueeze(1)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False
)

In [ ]:
model.eval()

test_probs = []
test_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        probs = torch.sigmoid(logits)

        test_probs.append(probs.cpu())
        test_targets.append(y_batch.cpu())

test_probs = torch.cat(test_probs).numpy().ravel()
test_targets = torch.cat(test_targets).numpy().ravel()

In [ ]:
chosen_threshold_pytorch = 0.93

test_preds = (
    test_probs >= chosen_threshold_pytorch
).astype(int)

In [ ]:
print("Confusion matrix:")
print(confusion_matrix(test_targets, test_preds))

print("Precision:", precision_score(test_targets, test_preds))
print("Recall:", recall_score(test_targets, test_preds))
print("F1:", f1_score(test_targets, test_preds))
print("ROC-AUC:", roc_auc_score(test_targets, test_probs))
print("PR-AUC:", average_precision_score(test_targets, test_probs))
print("Alerts:", test_preds.sum())

In [ ]:
import mlflow

In [ ]:
mlflow.set_experiment(
    "fraud-risk-model-comparison"
)

In [ ]:
with mlflow.start_run(run_name="test-run"):
    mlflow.log_param("model_type", "test")
    mlflow.log_metric("example_metric", 0.5)

In [ ]:
with mlflow.start_run(run_name="logistic-regression"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("threshold", 0.91)

    mlflow.log_metric("precision", 0.27435387673956263)
    mlflow.log_metric("recall", 0.19327731092436976)
    mlflow.log_metric("f1", 0.22678718159408381)
    mlflow.log_metric("roc_auc", 0.881467064633326)
    mlflow.log_metric("pr_auc", 0.1791398735758737)
    mlflow.log_metric("alerts", 1006)

In [ ]:
with mlflow.start_run(run_name="xgboost"):
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("threshold", 0.90)
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("scale_pos_weight", 96.5327)

    mlflow.log_metric("precision", 0.26487367563162184)
    mlflow.log_metric("recall", 0.22759103641456582)
    mlflow.log_metric("f1", 0.2448210922787194)
    mlflow.log_metric("roc_auc", 0.891849055819991)
    mlflow.log_metric("pr_auc", 0.190218605756902)
    mlflow.log_metric("alerts", 1227)

In [ ]:
with mlflow.start_run(run_name="pytorch-mlp"):
    mlflow.log_param("model_type", "PyTorch_MLP")
    mlflow.log_param("threshold", 0.93)
    mlflow.log_param("input_dim", 51)
    mlflow.log_param("hidden_layers", "64,32")
    mlflow.log_param("dropout_1", 0.3)
    mlflow.log_param("dropout_2", 0.2)
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("weight_decay", 1e-4)

    mlflow.log_metric("precision", 0.238230289279637)
    mlflow.log_metric("recall", 0.29411764705882354)
    mlflow.log_metric("f1", 0.26324036352240676)
    mlflow.log_metric("roc_auc", 0.8876885193106744)
    mlflow.log_metric("pr_auc", 0.18846827195048896)
    mlflow.log_metric("alerts", 1763)

In [ ]:
import mlflow

print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
for exp in mlflow.search_experiments():
    print(exp.experiment_id, exp.name, exp.artifact_location)

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-risk-model-comparison")

In [ ]:
with mlflow.start_run(run_name="logistic-regression"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("threshold", 0.91)

    mlflow.log_metric("precision", 0.27435387673956263)
    mlflow.log_metric("recall", 0.19327731092436976)
    mlflow.log_metric("f1", 0.22678718159408381)
    mlflow.log_metric("roc_auc", 0.881467064633326)
    mlflow.log_metric("pr_auc", 0.1791398735758737)
    mlflow.log_metric("alerts", 1006)

In [ ]:
with mlflow.start_run(run_name="xgboost"):
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("threshold", 0.90)
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("scale_pos_weight", 96.5327)

    mlflow.log_metric("precision", 0.26487367563162184)
    mlflow.log_metric("recall", 0.22759103641456582)
    mlflow.log_metric("f1", 0.2448210922787194)
    mlflow.log_metric("roc_auc", 0.891849055819991)
    mlflow.log_metric("pr_auc", 0.190218605756902)
    mlflow.log_metric("alerts", 1227)

In [ ]:
with mlflow.start_run(run_name="pytorch-mlp"):
    mlflow.log_param("model_type", "PyTorch_MLP")
    mlflow.log_param("threshold", 0.93)
    mlflow.log_param("input_dim", 51)
    mlflow.log_param("hidden_layer_1", 64)
    mlflow.log_param("hidden_layer_2", 32)
    mlflow.log_param("dropout_1", 0.3)
    mlflow.log_param("dropout_2", 0.2)
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("weight_decay", 1e-4)
    mlflow.log_param("batch_size", 512)
    mlflow.log_param("pos_weight", 96.5327)

    mlflow.log_metric("precision", 0.238230289279637)
    mlflow.log_metric("recall", 0.29411764705882354)
    mlflow.log_metric("f1", 0.26324036352240676)
    mlflow.log_metric("roc_auc", 0.8876885193106744)
    mlflow.log_metric("pr_auc", 0.18846827195048896)
    mlflow.log_metric("alerts", 1763)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

In [ ]:
fig, ax = plt.subplots()

ConfusionMatrixDisplay.from_predictions(
    test_targets,
    test_preds,
    ax=ax
)

plt.title("PyTorch MLP - Confusion Matrix")
plt.tight_layout()
plt.savefig("pytorch_confusion_matrix.png")
plt.close()

In [ ]:
fig, ax = plt.subplots()

RocCurveDisplay.from_predictions(
    test_targets,
    test_probs,
    ax=ax
)

plt.title("PyTorch MLP - ROC Curve")
plt.tight_layout()
plt.savefig("pytorch_roc_curve.png")
plt.close()

In [ ]:
fig, ax = plt.subplots()

PrecisionRecallDisplay.from_predictions(
    test_targets,
    test_probs,
    ax=ax
)

plt.title("PyTorch MLP - Precision-Recall Curve")
plt.tight_layout()
plt.savefig("pytorch_pr_curve.png")
plt.close()

In [ ]:
with mlflow.start_run(run_name="pytorch-mlp-artifacts"):
    mlflow.log_param("model_type", "PyTorch_MLP")
    mlflow.log_param("threshold", 0.93)

    mlflow.log_metric("precision", 0.238230289279637)
    mlflow.log_metric("recall", 0.29411764705882354)
    mlflow.log_metric("f1", 0.26324036352240676)
    mlflow.log_metric("roc_auc", 0.8876885193106744)
    mlflow.log_metric("pr_auc", 0.18846827195048896)

    mlflow.log_artifact("pytorch_confusion_matrix.png")
    mlflow.log_artifact("pytorch_roc_curve.png")
    mlflow.log_artifact("pytorch_pr_curve.png")

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

def save_model_artifacts(
    y_true,
    y_pred,
    y_proba,
    model_name,
    prefix
):
    # Confusion Matrix
    fig, ax = plt.subplots()

    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        ax=ax
    )

    ax.set_title(f"{model_name} - Confusion Matrix")
    fig.tight_layout()
    fig.savefig(f"{prefix}_confusion_matrix.png")
    plt.close(fig)

    # ROC Curve
    fig, ax = plt.subplots()

    RocCurveDisplay.from_predictions(
        y_true,
        y_proba,
        ax=ax
    )

    ax.set_title(f"{model_name} - ROC Curve")
    fig.tight_layout()
    fig.savefig(f"{prefix}_roc_curve.png")
    plt.close(fig)

    # Precision-Recall Curve
    fig, ax = plt.subplots()

    PrecisionRecallDisplay.from_predictions(
        y_true,
        y_proba,
        ax=ax
    )

    ax.set_title(f"{model_name} - Precision-Recall Curve")
    fig.tight_layout()
    fig.savefig(f"{prefix}_pr_curve.png")
    plt.close(fig)

In [ ]:
"baseline_model" in globals()

In [ ]:
print("LogReg:", "baseline_model" in globals())
print("XGBoost:", "xgb_pipeline" in globals())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    exclude=["object", "string"]
).columns.tolist()

logreg_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", logreg_preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
baseline_model.fit(X_train, y_train)

In [ ]:
logreg_test_proba = baseline_model.predict_proba(X_test)[:, 1]

logreg_test_pred = (
    logreg_test_proba >= 0.91
).astype(int)

In [ ]:
print(confusion_matrix(y_test, logreg_test_pred))
print("Precision:", precision_score(y_test, logreg_test_pred))
print("Recall:", recall_score(y_test, logreg_test_pred))
print("F1:", f1_score(y_test, logreg_test_pred))

In [ ]:
save_model_artifacts(
    y_test,
    logreg_test_pred,
    logreg_test_proba,
    "Logistic Regression",
    "logreg"
)

In [ ]:
with mlflow.start_run(run_name="logistic-regression-artifacts"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("threshold", 0.91)

    mlflow.log_metric("precision", 0.27435387673956263)
    mlflow.log_metric("recall", 0.19327731092436976)
    mlflow.log_metric("f1", 0.22678718159408381)
    mlflow.log_metric("roc_auc", 0.881467064633326)
    mlflow.log_metric("pr_auc", 0.1791398735758737)

    mlflow.log_artifact("logreg_confusion_matrix.png")
    mlflow.log_artifact("logreg_roc_curve.png")
    mlflow.log_artifact("logreg_pr_curve.png")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=96.5327,
    eval_metric="logloss",
    random_state=42
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("classifier", xgb_model)
    ]
)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

df = pd.read_csv("../data/raw/Base.csv")

In [ ]:
X_train = df.loc[
    df["month"].between(0, 5)
].drop(columns=["fraud_bool", "month"])

y_train = df.loc[
    df["month"].between(0, 5),
    "fraud_bool"
]

X_test = df.loc[
    df["month"] == 7
].drop(columns=["fraud_bool", "month"])

y_test = df.loc[
    df["month"] == 7,
    "fraud_bool"
]

In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

In [ ]:
xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                dtype=np.float32
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=96.5327,
    eval_metric="logloss",
    tree_method="hist",
    max_bin=256,
    n_jobs=4,
    random_state=42
)

In [ ]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("classifier", xgb_model)
    ]
)

In [ ]:
xgb_pipeline.fit(X_train, y_train)

In [ ]:
xgb_test_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

xgb_test_pred = (
    xgb_test_proba >= 0.90
).astype(int)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

print("Confusion matrix:")
print(confusion_matrix(y_test, xgb_test_pred))

print("Precision:", precision_score(y_test, xgb_test_pred))
print("Recall:", recall_score(y_test, xgb_test_pred))
print("F1:", f1_score(y_test, xgb_test_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_test_proba))
print("PR-AUC:", average_precision_score(y_test, xgb_test_proba))
print("Alerts:", xgb_test_pred.sum())

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

def save_model_artifacts(
    y_true,
    y_pred,
    y_proba,
    model_name,
    prefix
):
    # Confusion Matrix
    fig, ax = plt.subplots()
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        ax=ax
    )
    ax.set_title(f"{model_name} - Confusion Matrix")
    fig.tight_layout()
    fig.savefig(f"{prefix}_confusion_matrix.png")
    plt.close(fig)

    # ROC Curve
    fig, ax = plt.subplots()
    RocCurveDisplay.from_predictions(
        y_true,
        y_proba,
        ax=ax
    )
    ax.set_title(f"{model_name} - ROC Curve")
    fig.tight_layout()
    fig.savefig(f"{prefix}_roc_curve.png")
    plt.close(fig)

    # Precision-Recall Curve
    fig, ax = plt.subplots()
    PrecisionRecallDisplay.from_predictions(
        y_true,
        y_proba,
        ax=ax
    )
    ax.set_title(f"{model_name} - Precision-Recall Curve")
    fig.tight_layout()
    fig.savefig(f"{prefix}_pr_curve.png")
    plt.close(fig)

In [ ]:
save_model_artifacts(
    y_test,
    xgb_test_pred,
    xgb_test_proba,
    "XGBoost",
    "xgboost"
)

In [ ]:
xgb_pipeline.fit(X_train, y_train)

In [ ]:
import mlflow

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-risk-model-comparison")

In [ ]:
with mlflow.start_run(run_name="xgboost-artifacts"):
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("threshold", 0.90)

    mlflow.log_metric("precision", 0.26487367563162184)
    mlflow.log_metric("recall", 0.22759103641456582)
    mlflow.log_metric("f1", 0.2448210922787194)
    mlflow.log_metric("roc_auc", 0.891849055819991)
    mlflow.log_metric("pr_auc", 0.190218605756902)
    mlflow.log_metric("alerts", 1227)

    mlflow.log_artifact("xgboost_confusion_matrix.png")
    mlflow.log_artifact("xgboost_roc_curve.png")
    mlflow.log_artifact("xgboost_pr_curve.png")

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv("../data/raw/Base.csv")

In [ ]:
X_train = df.loc[
    df["month"].between(0, 5)
].drop(columns=["fraud_bool", "month"])

y_train = df.loc[
    df["month"].between(0, 5),
    "fraud_bool"
]

X_test = df.loc[
    df["month"] == 7
].drop(columns=["fraud_bool", "month"])

y_test = df.loc[
    df["month"] == 7,
    "fraud_bool"
]

In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

In [ ]:
xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                dtype=np.float32
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=96.5327,
    eval_metric="logloss",
    tree_method="hist",
    max_bin=256,
    n_jobs=4,
    random_state=42
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("classifier", xgb_model)
    ]
)

In [ ]:
xgb_pipeline.fit(X_train, y_train)

In [ ]:
xgb_test_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

xgb_test_pred = (
    xgb_test_proba >= 0.90
).astype(int)

In [ ]:
error_df = X_test.copy()

error_df["y_true"] = y_test.values
error_df["y_proba"] = xgb_test_proba
error_df["y_pred"] = xgb_test_pred

In [ ]:
def classify_result(row):
    if row["y_true"] == 1 and row["y_pred"] == 1:
        return "TP"
    elif row["y_true"] == 0 and row["y_pred"] == 0:
        return "TN"
    elif row["y_true"] == 0 and row["y_pred"] == 1:
        return "FP"
    else:
        return "FN"

error_df["result_type"] = error_df.apply(
    classify_result,
    axis=1
)

In [ ]:
error_df["result_type"].value_counts()

In [ ]:
numeric_features = X_test.select_dtypes(
    exclude=["object", "string"]
).columns.tolist()

fp_vs_tn = (
    error_df[
        error_df["result_type"].isin(["FP", "TN"])
    ]
    .groupby("result_type")[numeric_features]
    .mean()
    .T
)

fp_vs_tn["absolute_difference"] = (
    fp_vs_tn["FP"] - fp_vs_tn["TN"]
).abs()

fp_vs_tn.sort_values(
    "absolute_difference",
    ascending=False
).head(10)

In [ ]:
fn_vs_tp = (
    error_df[
        error_df["result_type"].isin(["FN", "TP"])
    ]
    .groupby("result_type")[numeric_features]
    .mean()
    .T
)

fn_vs_tp["absolute_difference"] = (
    fn_vs_tp["FN"] - fn_vs_tp["TP"]
).abs()

fn_vs_tp.sort_values(
    "absolute_difference",
    ascending=False
).head(10)

In [ ]:
feature = "payment_type"

fn_distribution = (
    error_df[error_df["result_type"] == "FN"][feature]
    .value_counts(normalize=True)
)

tp_distribution = (
    error_df[error_df["result_type"] == "TP"][feature]
    .value_counts(normalize=True)
)

payment_error_comparison = pd.DataFrame({
    "FN_pct": fn_distribution,
    "TP_pct": tp_distribution
}).fillna(0)

payment_error_comparison["difference"] = (
    payment_error_comparison["FN_pct"]
    - payment_error_comparison["TP_pct"]
)

payment_error_comparison.sort_values(
    "difference",
    ascending=False
)

In [ ]:
categorical_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]

categorical_error_results = []

for feature in categorical_features:
    fn_dist = (
        error_df[error_df["result_type"] == "FN"][feature]
        .value_counts(normalize=True)
    )

    tp_dist = (
        error_df[error_df["result_type"] == "TP"][feature]
        .value_counts(normalize=True)
    )

    comparison = pd.DataFrame({
        "FN_pct": fn_dist,
        "TP_pct": tp_dist
    }).fillna(0)

    comparison["difference"] = (
        comparison["FN_pct"] - comparison["TP_pct"]
    )

    comparison["feature"] = feature
    comparison["category"] = comparison.index

    categorical_error_results.append(
        comparison.reset_index(drop=True)
    )

categorical_error_df = pd.concat(
    categorical_error_results,
    ignore_index=True
)

categorical_error_df["absolute_difference"] = (
    categorical_error_df["difference"].abs()
)

categorical_error_df.sort_values(
    "absolute_difference",
    ascending=False
).head(15)

In [ ]:
fraud_cases = error_df[
    error_df["y_true"] == 1
].copy()

segment_recall = (
    fraud_cases
    .groupby("device_os")["y_pred"]
    .agg(["count", "sum"])
)

segment_recall["recall"] = (
    segment_recall["sum"] / segment_recall["count"]
)

segment_recall.sort_values(
    "recall",
    ascending=True
)

In [ ]:
def recall_by_segment(df, feature):
    fraud_cases = df[df["y_true"] == 1].copy()

    result = (
        fraud_cases
        .groupby(feature)["y_pred"]
        .agg(["count", "sum"])
    )

    result["recall"] = result["sum"] / result["count"]

    return result.sort_values("recall")

In [ ]:
for feature in [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]:
    print(f"\n--- {feature} ---")
    print(recall_by_segment(error_df, feature))

In [ ]:
import shap

In [ ]:
preprocessor_fitted = xgb_pipeline.named_steps["preprocessor"]
xgb_fitted = xgb_pipeline.named_steps["classifier"]

X_test_transformed = preprocessor_fitted.transform(X_test)

In [ ]:
X_shap = X_test_transformed[:2000]

In [ ]:
explainer = shap.TreeExplainer(xgb_fitted)

shap_values = explainer.shap_values(X_shap)

In [ ]:
feature_names = preprocessor_fitted.get_feature_names_out()

shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    plot_type="bar"
)

In [ ]:
fn_index = error_df[
    error_df["result_type"] == "FN"
].index[0]

In [ ]:
fn_case = X_test.loc[[fn_index]]

In [ ]:
fn_case_transformed = preprocessor_fitted.transform(fn_case)

In [ ]:
fn_shap_values = explainer.shap_values(fn_case_transformed)

In [ ]:
shap.plots.waterfall(
    shap.Explanation(
        values=fn_shap_values[0],
        base_values=explainer.expected_value,
        data=fn_case_transformed[0],
        feature_names=feature_names
    )
)

In [ ]:
tp_index = error_df[
    error_df["result_type"] == "TP"
].index[0]

In [ ]:
tp_case = X_test.loc[[tp_index]]

In [ ]:
tp_case_transformed = preprocessor_fitted.transform(tp_case)

In [ ]:
tp_shap_values = explainer.shap_values(tp_case_transformed)

In [ ]:
shap.plots.waterfall(
    shap.Explanation(
        values=tp_shap_values[0],
        base_values=explainer.expected_value,
        data=tp_case_transformed[0],
        feature_names=feature_names
    )
)

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-risk-model-comparison")

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-risk-model-comparison")

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

df = pd.read_csv("../data/raw/Base.csv")

X_train = df.loc[
    df["month"].between(0, 5)
].drop(columns=["fraud_bool", "month"])

y_train = df.loc[
    df["month"].between(0, 5),
    "fraud_bool"
]

categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                dtype=np.float32
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=96.5327,
    eval_metric="logloss",
    tree_method="hist",
    max_bin=256,
    n_jobs=4,
    random_state=42
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("classifier", xgb_model)
    ]
)

xgb_pipeline.fit(X_train, y_train)

In [ ]:
with mlflow.start_run(run_name="xgboost-champion-model") as run:
    model_info = mlflow.sklearn.log_model(
        sk_model=xgb_pipeline,
        name="model"
    )

    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

In [ ]:
with mlflow.start_run(run_name="xgboost-champion-model") as run:
    model_info = mlflow.sklearn.log_model(
        sk_model=xgb_pipeline,
        name="model",
        skops_trusted_types=[
            "xgboost.core.Booster",
            "xgboost.sklearn.XGBClassifier",
        ],
    )

    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name="fraud-risk-model"
)

print("Registered version:", registered_model.version)

In [ ]:
client.set_registered_model_alias(
    name="fraud-risk-model",
    alias="champion",
    version=registered_model.version
)

print("Champion alias set.")

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")

champion_model = mlflow.sklearn.load_model(
    "models:/fraud-risk-model@champion"
)

In [ ]:
sample = X_test.iloc[[0]]

In [ ]:
fraud_probability = champion_model.predict_proba(sample)[:, 1]

print("Fraud probability:", fraud_probability[0])

In [ ]:
X_train.columns.tolist()

In [ ]:
type(xgb_pipeline)

In [ ]:
type(xgb_pipeline.named_steps["classifier"])

In [ ]:
xgb_pipeline.named_steps

In [ ]:
xgb_model = xgb_pipeline.named_steps["classifier"]

xgb_model.save_model("xgb_champion.ubj")

In [ ]:
from xgboost import XGBClassifier

reloaded_xgb = XGBClassifier()
reloaded_xgb.load_model("xgb_champion.ubj")

print(type(reloaded_xgb))

In [ ]:
preprocessor = xgb_pipeline.named_steps["preprocessor"]

X_sample = X_test.iloc[:10]

X_sample_transformed = preprocessor.transform(X_sample)

original_probs = xgb_model.predict_proba(
    X_sample_transformed
)[:, 1]

reloaded_probs = reloaded_xgb.predict_proba(
    X_sample_transformed
)[:, 1]

print(original_probs)
print(reloaded_probs)

In [ ]:
import numpy as np

print(
    np.allclose(
        original_probs,
        reloaded_probs,
        rtol=1e-6,
        atol=1e-8,
    )
)

In [ ]:
import joblib

preprocessor = xgb_pipeline.named_steps["preprocessor"]

joblib.dump(
    preprocessor,
    "xgb_preprocessor.joblib",
)

In [ ]:
reloaded_preprocessor = joblib.load(
    "xgb_preprocessor.joblib"
)

X_sample = X_test.iloc[:10]

original_transformed = preprocessor.transform(X_sample)
reloaded_transformed = reloaded_preprocessor.transform(X_sample)

In [ ]:
import numpy as np

print(
    np.allclose(
        original_transformed,
        reloaded_transformed,
        rtol=1e-6,
        atol=1e-8,
    )
)

In [ ]:
import mlflow.pyfunc
import joblib
from xgboost import XGBClassifier


class FraudRiskPyFuncModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.preprocessor = joblib.load(
            context.artifacts["preprocessor"]
        )

        self.model = XGBClassifier()
        self.model.load_model(
            context.artifacts["xgb_model"]
        )

    def predict(self, context, model_input):
        transformed = self.preprocessor.transform(model_input)

        probabilities = self.model.predict_proba(
            transformed
        )[:, 1]

        return probabilities

In [ ]:
import mlflow

artifacts = {
    "preprocessor": "xgb_preprocessor.joblib",
    "xgb_model": "xgb_champion.ubj",
}

In [ ]:
with mlflow.start_run(
    run_name="xgboost-native-serving-candidate"
) as run:

    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=FraudRiskPyFuncModel(),
        artifacts=artifacts,
        pip_requirements=[
            "mlflow==3.15.1",
            "scikit-learn==1.9.0",
            "xgboost==3.4.1",
            "pandas==2.3.3",
            "numpy==2.5.2",
            "joblib",
        ],
    )

    print(model_info.model_uri)

In [ ]:
loaded_candidate = mlflow.pyfunc.load_model(
    model_info.model_uri
)

In [ ]:
X_sample = X_test.iloc[:10]

candidate_probs = loaded_candidate.predict(
    X_sample
)

original_probs = xgb_pipeline.predict_proba(
    X_sample
)[:, 1]

In [ ]:
import numpy as np

print(candidate_probs)
print(original_probs)

print(
    np.allclose(
        candidate_probs,
        original_probs,
        rtol=1e-6,
        atol=1e-8,
    )
)

In [ ]:
registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name="fraud-risk-model",
)

print("registered version:", registered.version)

In [ ]:
v2_model = mlflow.pyfunc.load_model(
    f"models:/fraud-risk-model/{registered.version}"
)

v2_probs = v2_model.predict(X_test.iloc[:10])

print(v2_probs)

print(
    np.allclose(
        v2_probs,
        original_probs,
        rtol=1e-6,
        atol=1e-8,
    )
)

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name="fraud-risk-model",
    alias="champion",
    version=registered.version,
)

In [ ]:
X_test.iloc[0].to_dict()

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/Base.csv")

test_df = df[df["month"] == 7].copy()

X_test = test_df.drop(
    columns=["fraud_bool", "month"]
)

y_test = test_df["fraud_bool"]

print(X_test.shape)
print(y_test.mean())

In [ ]:
import joblib
import numpy as np
from xgboost import XGBClassifier

preprocessor = joblib.load("xgb_preprocessor.joblib")

xgb = XGBClassifier()
xgb.load_model("xgb_champion.ubj")

batch = X_test.iloc[:1000]

X_batch = preprocessor.transform(batch)
probs = xgb.predict_proba(X_batch)[:, 1]

matches = np.where(probs >= 0.90)[0]

print("numar cazuri >= 0.90:", len(matches))

idx = matches[0]

print("index:", idx)
print("probability:", probs[idx])
print(X_test.iloc[idx].to_dict())